# 적용한 내용
[llm 객체를 생성](#llm-prompt-chain-생성-및-작동-확인)할 때, max_tokens 파라미터에 200을 인자로 주어 답변 길이를 제어하였다.

# 실행 환경 감지

In [1]:
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


# API 키 로드

In [2]:
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


# 패키지 임포트

In [3]:
# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")

✅ import 완료


# llm, prompt, chain 생성 및 작동 확인

In [ ]:
# max_tokens=200: 모든 답변을 200 토큰 이내로 제한 (토큰 폭주 방지 + 비용 관리)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=300)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{question}")
])

chain = prompt | llm   # LCEL 파이프 연산자 (2교시 3부)

# 동작 확인
result = chain.invoke({"role": "데이터 분석가", "question": "이상치란 무엇인가요? 한 문장으로."})
print(result.content)

이상치는 데이터 집합에서 다른 값들과


# 페르소나 사전 생성

In [11]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}

print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


# 챗봇 함수 정의

In [12]:
# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 이름으로 system 메시지 조회
    system_message = PERSONAS[persona_name]

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})

    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

In [13]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

# 챗봇 실행 함수 정의

In [14]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

# 챗봇 실행

In [15]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 페르소나(Persona)는 특정 사용자 그룹

👋 종료합니다.
